In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import yaml
import os
import glob

## Input Parameters

In [ ]:
# Path to the Pynta calculation directory (contains TS0, TS1, ... folders)
base_path = '/Users/shinae/pynta-production/for_shinae_hb/nh3-vdw-pt111-forMatt_hb_yesdiffusion_vib'

# Path to the reaction yaml file
yaml_path = '/Users/shinae/pynta-production/for_shinae_hb/nh3-vdw-pt111-forMatt_hb_yesdiffusion_vib/hb-yesdiffusion.yaml'

# Select which TS directories to plot, e.g. ['TS0', 'TS2', 'TS5']
selected_ts = ['TS0', 'TS1', 'TS2']

# Temperature range (K)
T = np.linspace(500, 1400, 50)

## Load Data

In [ ]:
def parse_surface_arrhenius(text):
    # Parses: SurfaceArrhenius(A=(val,'units'), n=val, Ea=(val,'units'), T0=(1,'K'))
    a_str    = text.split('A=(')[1].split(')')[0]
    n_str    = text.split(', n=')[1].split(',')[0].strip()
    ea_str   = text.split('Ea=(')[1].split(')')[0]
    A_val    = float(a_str.split(',')[0])
    A_units  = a_str.split(',', 1)[1].strip().replace(chr(39), '')
    n_val    = float(n_str)
    Ea_val   = float(ea_str.split(',')[0])
    Ea_units = ea_str.split(',', 1)[1].strip().replace(chr(39), '')
    Ea_kJ    = Ea_val if 'kJ' in Ea_units else Ea_val / 1000.0
    return A_val, n_val, Ea_kJ, A_units


def load_kinetics_log(path):
    result = {}
    with open(path) as f:
        for line in f:
            line = line.strip()
            if line.startswith('Reaction:'):
                result['reaction'] = line.split('Reaction:', 1)[1].strip()
            elif line.startswith('arr_f:'):
                val = line.split('arr_f:', 1)[1].strip()
                result['arr_f'] = None if val == 'None' else parse_surface_arrhenius(val)
            elif line.startswith('arr_r:'):
                val = line.split('arr_r:', 1)[1].strip()
                result['arr_r'] = None if val == 'None' else parse_surface_arrhenius(val)
    return result


# Load yaml: map reaction string -> entry
with open(yaml_path) as f:
    yaml_entries = yaml.safe_load(f)
yaml_reactions = {e['reaction']: e for e in yaml_entries}

# For each selected TS find all kinetics.log files; keep the one with lowest forward Ea
ts_data = {}
for ts_name in selected_ts:
    ts_dir    = os.path.join(base_path, ts_name)
    log_files = sorted(glob.glob(os.path.join(ts_dir, '*', 'kinetics.log')))
    if not log_files:
        print('{}: no kinetics.log found. Run Pynta Postprocessing notebook first.'.format(ts_name))
        continue
    best, best_ea = None, float('inf')
    for lf in log_files:
        d = load_kinetics_log(lf)
        if d.get('arr_f') is not None and d['arr_f'][2] < best_ea:
            best_ea = d['arr_f'][2]
            best    = d
    if best:
        ts_data[ts_name] = best
        print('{}: {}  (Ea_f = {:.1f} kJ/mol)'.format(ts_name, best.get('reaction', 'unknown'), best_ea))

## Calculate Rate Coefficients

In [ ]:
R = 0.0083145  # kJ/(mol*K)

records_f, records_r = [], []

for ts_name, data in ts_data.items():
    rxn_str = data.get('reaction', ts_name)
    label   = yaml_reactions.get(rxn_str, {}).get('reaction', rxn_str)

    if data.get('arr_f') is not None:
        A, n, Ea_kJ, A_units = data['arr_f']
        k = A * (T ** n) * np.exp(-Ea_kJ / (R * T))
        records_f.append({'label': label, 'k': k, 'A_units': A_units})

    if data.get('arr_r') is not None:
        A, n, Ea_kJ, A_units = data['arr_r']
        k = A * (T ** n) * np.exp(-Ea_kJ / (R * T))
        records_r.append({'label': label, 'k': k, 'A_units': A_units})

## Plot Rate Coefficients

In [ ]:
tableau_colors = [
    '#1f77b4', '#ff7f0e', '#2ca02c', '#d62728', '#9467bd',
    '#8c564b', '#e377c2', '#7f7f7f', '#bcbd22', '#17becf'
]
line_thickness = 4


def style_ax(ax, ylabel):
    ax.set_yscale('log')
    ax.set_xlabel('Temperature (K)', fontsize=15, fontweight='bold')
    ax.set_ylabel(ylabel, fontsize=15, fontweight='bold')
    ax.set_xlim(T[0], T[-1])
    ax.tick_params(axis='both', which='major', labelsize=13, width=2)
    for tick in ax.xaxis.get_major_ticks():
        tick.label1.set_fontweight('bold')
    for tick in ax.yaxis.get_major_ticks():
        tick.label1.set_fontweight('bold')
    for spine in ax.spines.values():
        spine.set_linewidth(3)
    ax.grid(False)
    ax.legend(fontsize=10, loc='best')


# Forward rate coefficients
fig, ax = plt.subplots(figsize=(8, 7))
for i, rec in enumerate(records_f):
    ax.plot(T, rec['k'], label=rec['label'],
            color=tableau_colors[i % len(tableau_colors)], linewidth=line_thickness)
style_ax(ax, 'Rate Coefficient k (forward)')
plt.tight_layout()
plt.show()


# Reverse rate coefficients
if records_r:
    fig, ax = plt.subplots(figsize=(8, 7))
    for i, rec in enumerate(records_r):
        ax.plot(T, rec['k'], label=rec['label'],
                color=tableau_colors[i % len(tableau_colors)],
                linewidth=line_thickness, linestyle='--')
    style_ax(ax, 'Rate Coefficient k (reverse)')
    plt.tight_layout()
    plt.show()